read from Bronze table, unify naming and category type, filter by rules, connect dimension tables and write into Silver table

In [0]:
from functools import reduce
from pyspark.sql import functions as F

BRONZE_TABLE = "nyc_taxi.bronze.yellow_trips"
ZONE_TABLE   = "nyc_taxi.bronze.taxi_zone_lookup"
SILVER_TABLE = "nyc_taxi.silver.yellow_trips"

START_DATE = "2019-01-01"   # 包含
END_DATE   = "2019-04-01"   # 不包含：[START_DATE, END_DATE)

In [0]:
typed = spark.table(BRONZE_TABLE).select(
    F.col("VendorID").cast("int").alias("vendor_id"),
    F.col("tpep_pickup_datetime").alias("pickup_datetime"),
    F.col("tpep_dropoff_datetime").alias("dropoff_datetime"),
    F.col("passenger_count").cast("int").alias("passenger_count"),
    F.col("trip_distance"),
    F.col("RatecodeID").cast("int").alias("rate_code_id"),
    F.col("store_and_fwd_flag"),
    F.col("PULocationID").cast("int").alias("pickup_location_id"),
    F.col("DOLocationID").cast("int").alias("dropoff_location_id"),
    F.col("payment_type").cast("int").alias("payment_type"),
    F.col("fare_amount"),
    F.col("extra"),
    F.col("mta_tax"),
    F.col("tip_amount"),
    F.col("tolls_amount"),
    F.col("improvement_surcharge"),
    F.col("total_amount"),
    F.col("congestion_surcharge"),
    F.col("_source_file"),
    F.col("_ingested_at"),
).withColumns({
    "pickup_date":  F.to_date("pickup_datetime"),
    "pickup_year":  F.year("pickup_datetime"),
    "pickup_month": F.month("pickup_datetime"),
    "trip_duration_min": F.expr("timestampdiff(SECOND, pickup_datetime, dropoff_datetime)") / 60,
})

In [0]:
RULES = {
    "trip_distance > 0":   F.col("trip_distance") > 0,
    "fare_amount >= 0":    F.col("fare_amount") >= 0,
    "total_amount > 0":    F.col("total_amount") > 0,
    "passenger_count > 0": F.col("passenger_count") > 0,
    "pickup not null":     F.col("pickup_datetime").isNotNull(),
    "dropoff not null":    F.col("dropoff_datetime").isNotNull(),
    "dropoff > pickup":    F.col("dropoff_datetime") > F.col("pickup_datetime"),
    "pickup in range":     (F.col("pickup_date") >= START_DATE) & (F.col("pickup_date") < END_DATE),
}

display(typed.select([
    F.sum(F.when(F.coalesce(cond, F.lit(False)), 0).otherwise(1)).alias(name)
    for name, cond in RULES.items()
]))

In [0]:
is_valid = reduce(lambda a, b: a & b, RULES.values())
cleaned = typed.filter(is_valid)

In [0]:
zones = spark.table(ZONE_TABLE)

pu_zone = zones.select(
    F.col("LocationID").cast("int").alias("pickup_location_id"),
    F.col("Borough").alias("pickup_borough"),
    F.col("Zone").alias("pickup_zone"),
)
do_zone = zones.select(
    F.col("LocationID").cast("int").alias("dropoff_location_id"),
    F.col("Borough").alias("dropoff_borough"),
    F.col("Zone").alias("dropoff_zone"),
)

RATECODE_ROWS = [
    (1, "Standard rate"), (2, "JFK"), (3, "Newark"),
    (4, "Nassau or Westchester"), (5, "Negotiated fare"), (6, "Group ride"),
]
ratecode = spark.createDataFrame(RATECODE_ROWS, "rate_code_id int, rate_code_desc string")

silver_df = (
    cleaned
    .join(F.broadcast(pu_zone),  on="pickup_location_id",  how="left")
    .join(F.broadcast(do_zone),  on="dropoff_location_id", how="left")
    .join(F.broadcast(ratecode), on="rate_code_id",        how="left")
)

In [0]:
from pyspark.sql import Window

KEY = ["vendor_id", "pickup_datetime", "dropoff_datetime",
       "pickup_location_id", "dropoff_location_id"]
BUSINESS_COLS = [c for c in silver_df.columns if not c.startswith("_")]

# --- 1) 去掉完全重复：业务列全部相同才算重复 ---
rows_before_dedup = silver_df.count()
silver_df = silver_df.dropDuplicates(BUSINESS_COLS)

# --- 2) 业务键冲突的行：组内按全部非键列排序，生成确定的序号 ---
order_cols = [c for c in BUSINESS_COLS if c not in KEY]
w = Window.partitionBy(*KEY).orderBy(*[F.col(c).asc_nulls_last() for c in order_cols])

silver_df = (silver_df
    .withColumn("trip_seq", F.row_number().over(w))
    .withColumn("trip_id", F.sha2(F.concat_ws("||",
        *[F.coalesce(F.col(c).cast("string"), F.lit("<NULL>")) for c in KEY + ["trip_seq"]]), 256)))

# trip_id 放在第一列，方便阅读
silver_df = silver_df.select("trip_id", *[c for c in silver_df.columns if c != "trip_id"])

rows_after_dedup = silver_df.count()
print(f"去重前 {rows_before_dedup:,} → 去重后 {rows_after_dedup:,}，删除 {rows_before_dedup - rows_after_dedup:,}")

In [0]:
(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TABLE)
)

In [0]:
%sql
COMMENT ON TABLE nyc_taxi.silver.yellow_trips IS
  'Cleaned NYC yellow taxi trips 2019-01..03: snake_case names, explicit types, quality rules applied, pickup in [2019-01-01, 2019-04-01), zone and rate code enriched.';

In [0]:
bronze_count = spark.table(BRONZE_TABLE).count()
silver_count = spark.table(SILVER_TABLE).count()
dropped = bronze_count - silver_count

print(f"bronze : {bronze_count:,}")
print(f"silver : {silver_count:,}   (stage 1 migration, without date rule: 21,966,573)")
print(f"dropped: {dropped:,}  ratio {dropped / bronze_count:.2%}")

display(
    spark.table(SILVER_TABLE)
    .groupBy("pickup_year", "pickup_month").count()
    .orderBy("pickup_year", "pickup_month")
)

In [0]:
s = spark.table(SILVER_TABLE)
total = s.count()
uniq = s.select("trip_id").distinct().count()
print(f"总行数 {total:,} / trip_id 唯一数 {uniq:,}")
assert total == uniq, "trip_id 不唯一！"
display(s.groupBy("trip_seq").count().orderBy("trip_seq"))